In [21]:
!pip install xgboost lightgbm catboost pandas numpy scikit-learn

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             mean_squared_error, r2_score)
from sklearn.model_selection import StratifiedKFold, KFold, cross_validate

In [23]:
ownership_threshold_log = 10
imp_threshold = 0.003
corr_threshold = 0.90
verbose = False

In [24]:
# Code for correlation analysis and feature selection

import numpy as np

def get_top_correlations(df, top_n=30, threshold=0.90, verbose=False):
    corr_matrix = df.corr().abs()  # Get absolute correlation values
    corr_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))  # upper triangle only
    .stack()
    .reset_index()
    )   
    corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
    corr_pairs = corr_pairs.sort_values('Correlation', ascending=False)

    if verbose:
        print("Top 30 most correlated feature pairs:")
        print(corr_pairs.head(30).to_string(index=False))
    
    to_drop = get_highly_correlated_features(corr_pairs, threshold=threshold, verbose=verbose)

    return to_drop

    
def get_highly_correlated_features(corr_pairs, threshold=0.90, verbose=False):
    to_drop = set()

    for _, row in corr_pairs.iterrows():
        f1, f2, corr = row['Feature 1'], row['Feature 2'], row['Correlation']
        if corr < threshold:
            break  # already sorted descending, so we can stop early
        # Drop f2 (keep f1) — unless f1 is already marked for dropping
        if f1 not in to_drop:
            to_drop.add(f2)

    if verbose:
        print(f"Features to drop ({len(to_drop)}):")
        print(sorted(to_drop))
    
    return to_drop

In [25]:
# load data and importance scores
df_final = pd.read_csv('./cleaned_dataset/steam_finalized_dataset.csv')
importance_df = pd.read_csv('./cleaned_dataset/feature_importances.csv')
df_final = df_final[df_final['owners_log'] >= ownership_threshold_log]
df_final = df_final[df_final['price'] > 0]

print(f"Dataset shape after filtering: {df_final.shape}")

Dataset shape after filtering: (6845, 499)


In [26]:
# feature selection — lower threshold to capture more tag/genre signal
selected_features = importance_df[
    importance_df['importance'] > imp_threshold
]['feature'].tolist()

# # playtime sits just below threshold but is informative? 
# for col in ['average_playtime', 'median_playtime']:
#     if col in df_final.columns and col not in selected_features:
#         selected_features.append(col)

# Label-encode string columns so .corr() and feature selection work
for col in ["developer", "publisher"]:
    if col in df_final.columns and df_final[col].dtype == object:
        df_final[col] = pd.factorize(df_final[col])[0]

print(f"Selected {len(selected_features)} features (importance > {imp_threshold})")

selected_features = [col for col in selected_features if col not in ['publisher']]
print(selected_features)

Selected 40 features (importance > 0.003)
['owners_log', 'achievements', 'indie', 'price', 'singleplayer', 'developer', 'storage_mb', 'support_email', 'website', 'release_month', 'processor_Ghz', 'support_url', 'GPU_mb', 'RAM_mb', 'anime', 'adventure', 'casual', 'vr', 'Steam Cloud_cat', 'action', 'release_year', 'visual_novel', 'puzzle', 'strategy', 'great_soundtrack', 'simulation', 'Steam Trading Cards_cat', 'pixel_graphics', 'rpg', 'median_playtime', '2d', 'average_playtime', 'Full controller support_cat', 'early_access', 'mac', 'classic', 'Partial Controller Support_cat', 'mac_sup', 'horror']


In [27]:
le = LabelEncoder()   # 5-class encoder
le3 = LabelEncoder()  # 3-class encoder
n=30

to_drop = get_top_correlations(df_final[selected_features], top_n=30, threshold=corr_threshold, verbose=verbose)

X = df_final[selected_features].drop(columns=to_drop)
y_reg = df_final['wilson_score']
y_clf = le.fit_transform(df_final['rating_category'])

# collapsed 3-class target: Bad / Mixed / Good
merge_map = {
    'Negative':        'Bad',
    'Mostly Negative': 'Bad',
    'Mixed':           'Mixed',
    'Mostly Positive': 'Good',
    'Positive':        'Good',
}
y_clf3 = le3.fit_transform(df_final['rating_category'].map(merge_map))

X_train, X_test, y_clf_train, y_clf_test, y_clf3_train, y_clf3_test, y_reg_train, y_reg_test = train_test_split(
    X, y_clf, y_clf3, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

print("Train distribution (5-class):")
print(pd.Series(le.inverse_transform(y_clf_train)).value_counts(normalize=True).round(3))
print("\nTest distribution (5-class):")
print(pd.Series(le.inverse_transform(y_clf_test)).value_counts(normalize=True).round(3))
print("\nTrain distribution (3-class):")
print(pd.Series(le3.inverse_transform(y_clf3_train)).value_counts(normalize=True).round(3))

Train distribution (5-class):
Positive           0.342
Mostly Positive    0.284
Mixed              0.243
Mostly Negative    0.110
Negative           0.021
Name: proportion, dtype: float64

Test distribution (5-class):
Positive           0.343
Mostly Positive    0.284
Mixed              0.243
Mostly Negative    0.110
Negative           0.021
Name: proportion, dtype: float64

Train distribution (3-class):
Good     0.626
Mixed    0.243
Bad      0.131
Name: proportion, dtype: float64


In [28]:
# Raw versions 
X_train_raw = X_train.copy()
X_test_raw  = X_test.copy()

In [29]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

class TargetEncoderCV(BaseEstimator, TransformerMixin):
    """Target encodes a column using only the fold's training data."""
    def __init__(self, col):
        self.col = col
        
    def fit(self, X, y):
        X_ = X.copy() if hasattr(X, 'copy') else pd.DataFrame(X)
        self.enc_map_    = pd.Series(y).groupby(
            X_[self.col] if isinstance(X_, pd.DataFrame) else X_[:, self.col]
        ).mean()
        self.global_mean_ = float(pd.Series(y).mean())
        return self
    
    def transform(self, X):
        X_ = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        X_['developer_enc'] = X_[self.col].map(self.enc_map_).fillna(self.global_mean_)
        X_.drop(columns=[self.col], inplace=True)
        return X_


def make_clf_pipeline(model, scale=False):
    steps = [('target_enc', TargetEncoderCV(col='developer'))]
    # steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

def make_reg_pipeline(model, scale=False):
    steps = [('target_enc', TargetEncoderCV(col='developer'))]
    # steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

def make_poly_pipeline(model, degree=2, scale=True):
    steps = [('target_enc', TargetEncoderCV(col='developer'))]
    if scale:
        steps.append(('scaler', StandardScaler()))  # scale FIRST so poly terms stay bounded
    steps.append(('poly', PolynomialFeatures(degree=degree, include_bias=False)))
    steps.append(('model', model))
    return Pipeline(steps)

In [30]:
# Cross validation setup
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

In [45]:
from sklearn.preprocessing import PolynomialFeatures
deg = 3

clf_pipelines = {
    'Naive Bayes':         make_clf_pipeline(GaussianNB(), scale=True),
    'Logistic Regression': make_clf_pipeline(
        LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'), scale=True),
    'Random Forest':       make_clf_pipeline(
        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')),
    'XGBoost':             make_clf_pipeline(
        XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                      random_state=42, eval_metric='mlogloss', verbosity=0)),
    # 'LightGBM':            make_clf_pipeline(
    #     LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
    #                    subsample=0.8, colsample_bytree=0.8,
    #                    class_weight='balanced', random_state=42, verbose=-1)),
    'KNN':                 make_clf_pipeline(KNeighborsClassifier(n_neighbors=10, n_jobs=-1), scale=True),
}

clf3_pipelines = {
    'Logistic Regression': make_clf_pipeline(
        LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'), scale=True),
    'Random Forest':       make_clf_pipeline(
        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')),
    'XGBoost':             make_clf_pipeline(
        XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                      random_state=42, eval_metric='mlogloss', verbosity=0)),
    'KNN':                 make_clf_pipeline(KNeighborsClassifier(n_neighbors=10, n_jobs=-1), scale=True),
    # 'LightGBM':            make_clf_pipeline(
    #     LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
    #                    subsample=0.8, colsample_bytree=0.8,
    #                    class_weight='balanced', random_state=42, verbose=-1)),
}

reg_pipelines = {
    'Linear Regression': make_reg_pipeline(LinearRegression(), scale=True),
    'Polynomial Regression': make_poly_pipeline(Ridge(alpha=1000.0), degree=deg, scale=True),
    'Random Forest':     make_reg_pipeline(
        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    'XGBoost':           make_reg_pipeline(
        XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8,
                     random_state=42, verbosity=0)),
}

In [46]:
# CLASSIFICATION (5-class) — 5-Fold Stratified CV
print("CLASSIFICATION 5-CLASS — 5-Fold Stratified CV")
print("=" * 60)

clf_cv_results = {}
for name, pipe in clf_pipelines.items():
    scores = cross_validate(
        pipe, X_train_raw, y_clf_train,
        cv=cv_clf,
        scoring=['accuracy', 'f1_weighted'],
        n_jobs=-1
    )
    clf_cv_results[name] = {
        'CV Acc (mean)': scores['test_accuracy'].mean(),
        'CV Acc (std)':  scores['test_accuracy'].std(),
        'CV F1 (mean)':  scores['test_f1_weighted'].mean(),
        'CV F1 (std)':   scores['test_f1_weighted'].std(),
    }
    print(f"{name:22s} | Acc = {scores['test_accuracy'].mean():.4f} "
          f"± {scores['test_accuracy'].std():.4f} | "
          f"F1 = {scores['test_f1_weighted'].mean():.4f} "
          f"± {scores['test_f1_weighted'].std():.4f}")

# CLASSIFICATION (3-class) — 5-Fold Stratified CV
print("\nCLASSIFICATION 3-CLASS (Bad/Mixed/Good) — 5-Fold Stratified CV")
print("=" * 60)

cv_clf3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf3_cv_results = {}
for name, pipe in clf3_pipelines.items():
    scores = cross_validate(
        pipe, X_train_raw, y_clf3_train,
        cv=cv_clf3,
        scoring=['accuracy', 'f1_weighted'],
        n_jobs=-1
    )
    clf3_cv_results[name] = {
        'CV Acc (mean)': scores['test_accuracy'].mean(),
        'CV Acc (std)':  scores['test_accuracy'].std(),
        'CV F1 (mean)':  scores['test_f1_weighted'].mean(),
        'CV F1 (std)':   scores['test_f1_weighted'].std(),
    }
    print(f"{name:22s} | Acc = {scores['test_accuracy'].mean():.4f} "
          f"± {scores['test_accuracy'].std():.4f} | "
          f"F1 = {scores['test_f1_weighted'].mean():.4f} "
          f"± {scores['test_f1_weighted'].std():.4f}")

CLASSIFICATION 5-CLASS — 5-Fold Stratified CV
Naive Bayes            | Acc = 0.1775 ± 0.0120 | F1 = 0.2184 ± 0.0135
Logistic Regression    | Acc = 0.3817 ± 0.0159 | F1 = 0.4058 ± 0.0177
Random Forest          | Acc = 0.4766 ± 0.0145 | F1 = 0.4653 ± 0.0142
XGBoost                | Acc = 0.4936 ± 0.0117 | F1 = 0.4877 ± 0.0120
KNN                    | Acc = 0.3871 ± 0.0081 | F1 = 0.3880 ± 0.0095

CLASSIFICATION 3-CLASS (Bad/Mixed/Good) — 5-Fold Stratified CV
Logistic Regression    | Acc = 0.5643 ± 0.0171 | F1 = 0.5915 ± 0.0152
Random Forest          | Acc = 0.6899 ± 0.0088 | F1 = 0.6525 ± 0.0137
XGBoost                | Acc = 0.6872 ± 0.0055 | F1 = 0.6695 ± 0.0053
KNN                    | Acc = 0.6156 ± 0.0058 | F1 = 0.5840 ± 0.0053


In [49]:
def adjusted_r2(r2, n_samples, n_features):
    """Adjusted R² from an already-computed R² value."""
    return 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)

In [50]:
# REGRESSION — 5-Fold CV
print("REGRESSION — 5-Fold KFold CV")
print("=" * 50)

reg_cv_results = {}
for name, pipe in reg_pipelines.items():
    scores = cross_validate(
        pipe, X_train_raw, y_reg_train,
        cv=cv_reg,
        scoring=['r2', 'neg_root_mean_squared_error'],
        n_jobs=-1
    )
    reg_cv_results[name] = {
        'CV R² (mean)':   scores['test_r2'].mean(),
        'CV R² (std)':    scores['test_r2'].std(),
        'CV Adj R² (mean)': adjusted_r2(
            r2=scores['test_r2'].mean(),
            n_samples=len(y_reg_train) // 5,   # held-out fold size
            n_features=X_train_raw.shape[1]
        ),
        'CV RMSE (mean)': -scores['test_neg_root_mean_squared_error'].mean(),
        'CV RMSE (std)':  scores['test_neg_root_mean_squared_error'].std(),
    }
    print(f"{name:22s} | R² = {scores['test_r2'].mean():.4f} "
          f"Adjusted R² = {reg_cv_results[name]['CV Adj R² (mean)']:.4f} "
          f"± {scores['test_r2'].std():.4f} | "
          f"RMSE = {-scores['test_neg_root_mean_squared_error'].mean():.4f} "
          f"± {scores['test_neg_root_mean_squared_error'].std():.4f}")

REGRESSION — 5-Fold KFold CV
Linear Regression      | R² = 0.1934 Adjusted R² = 0.8540 ± 0.0116 | RMSE = 0.1814 ± 0.0039
Polynomial Regression  | R² = -2235.4920 Adjusted R² = 0.8884 ± 4100.8775 | RMSE = 5.4704 ± 7.5797
Random Forest          | R² = 0.2210 Adjusted R² = 0.9781 ± 0.0172 | RMSE = 0.1782 ± 0.0041
XGBoost                | R² = 0.2819 Adjusted R² = 0.9788 ± 0.0151 | RMSE = 0.1711 ± 0.0040


In [ ]:
print("\nFINAL TEST SET — CLASSIFICATION (5-class)")
print("=" * 50)

clf_test_results = {}
for name, pipe in clf_pipelines.items():
    pipe.fit(X_train_raw, y_clf_train)
    preds = pipe.predict(X_test_raw)
    acc = accuracy_score(y_clf_test, preds)
    clf_test_results[name] = acc
    print(f"\n{name} — Test Accuracy: {acc:.4f}")
    print(classification_report(y_clf_test, preds, target_names=le.classes_))

print("\nFINAL TEST SET — CLASSIFICATION (3-class: Bad/Mixed/Good)")
print("=" * 50)

clf3_test_results = {}
for name, pipe in clf3_pipelines.items():
    pipe.fit(X_train_raw, y_clf3_train)
    preds = pipe.predict(X_test_raw)
    acc = accuracy_score(y_clf3_test, preds)
    clf3_test_results[name] = acc
    print(f"\n{name} — Test Accuracy: {acc:.4f}")
    print(classification_report(y_clf3_test, preds, target_names=le3.classes_))

print("\nFINAL TEST SET — REGRESSION")
print("=" * 50)

reg_test_results = {}
for name, pipe in reg_pipelines.items():
    pipe.fit(X_train_raw, y_reg_train)
    preds = pipe.predict(X_test_raw)
    r2   = r2_score(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    reg_test_results[name] = {'R²': r2, 'RMSE': rmse}
    print(f"{name:22s} | R² = {r2:.4f} | RMSE = {rmse:.4f}")

In [ ]:
# Summary Tables
print("\nCLASSIFICATION SUMMARY (5-class)")
clf_summary = pd.DataFrame(clf_cv_results).T
clf_summary['Test Accuracy'] = pd.Series(clf_test_results)
print(clf_summary.sort_values('CV Acc (mean)', ascending=False).round(4))

print("\nCLASSIFICATION SUMMARY (3-class: Bad/Mixed/Good)")
clf3_summary = pd.DataFrame(clf3_cv_results).T
clf3_summary['Test Accuracy'] = pd.Series(clf3_test_results)
print(clf3_summary.sort_values('CV Acc (mean)', ascending=False).round(4))

print("\nREGRESSION SUMMARY")
reg_summary = pd.DataFrame(reg_cv_results).T
reg_summary['Test R²']   = pd.Series({k: v['R²']   for k, v in reg_test_results.items()})
reg_summary['Test RMSE'] = pd.Series({k: v['RMSE'] for k, v in reg_test_results.items()})
print(reg_summary.sort_values('CV R² (mean)', ascending=False).round(4))